# 🚀 DeepMIMO O1_28 Dataset Downloader
**For: 6G Beam Prediction Project | VIT Chennai | BECE311L**

This notebook downloads the `o1_28` (Outdoor 1, 28 GHz mmWave) DeepMIMO scenario
and saves it to your Google Drive so you can copy it to your local machine.

**Steps:**
1. Run all cells top to bottom (`Runtime → Run all`)
2. Authorize Google Drive when prompted
3. After completion, find the zip at `My Drive/DeepMIMO/o1_28_downloaded.zip`
4. Download it from Drive to your PC
5. Follow the extraction steps in the final cell

## Step 1 — Install DeepMIMO v4

In [ ]:
!pip install -q deepmimo==4.0.0b11
import deepmimo as dm
print(f'✓ DeepMIMO version: {dm.__version__}')

## Step 2 — Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
save_dir = '/content/drive/MyDrive/DeepMIMO'
os.makedirs(save_dir, exist_ok=True)
print(f'✓ Google Drive mounted. Files will be saved to: {save_dir}')

## Step 3 — Download o1_28 Scenario (~200–400 MB)

In [ ]:
import os, shutil

SCENARIO = 'o1_28'
SAVE_DIR = '/content/drive/MyDrive/DeepMIMO'

# Check if already on Drive
drive_zip = os.path.join(SAVE_DIR, f'{SCENARIO}_downloaded.zip')
drive_folder = os.path.join(SAVE_DIR, SCENARIO)

if os.path.exists(drive_folder):
    print(f'✓ Scenario folder already on Drive: {drive_folder}')
elif os.path.exists(drive_zip):
    print(f'✓ Zip already on Drive: {drive_zip}')
    print('  Skipping download — just extract it following the instructions below.')
else:
    # Download to Colab's local /tmp first (fast SSD)
    print(f'Downloading {SCENARIO} to Colab local storage first...')
    local_zip = dm.download(SCENARIO, output_dir='/tmp')
    
    if local_zip and os.path.exists(local_zip):
        print(f'\n✓ Download complete! Copying to Google Drive...')
        shutil.copy2(local_zip, drive_zip)
        print(f'✓ Saved to Drive: {drive_zip}')
        file_size_mb = os.path.getsize(drive_zip) / (1024**2)
        print(f'  File size: {file_size_mb:.1f} MB')
    else:
        print('\n⚠️  Download via dm.download() failed. Trying direct wget...')
        # Fallback: direct wget from known URL pattern
        import requests
        resp = requests.get(
            'https://deepmimo.net/api/download/secure',
            params={'filename': f'{SCENARIO}.zip'},
            headers={'User-Agent': 'DeepMIMO-Python/4.0'}
        )
        print(f'  API response status: {resp.status_code}')
        if resp.status_code == 200:
            data = resp.json()
            redirect_url = data.get('redirectUrl', '')
            if redirect_url:
                if not redirect_url.startswith('http'):
                    redirect_url = 'https://deepmimo.net' + redirect_url
                print(f'  Downloading from: {redirect_url[:80]}...')
                !wget -q --show-progress -O "{drive_zip}" "{redirect_url}"
                print(f'✓ Downloaded to {drive_zip}')
        else:
            print(f'  Status: {resp.status_code} — {resp.text[:200]}')
            print()
            print('❌ Both methods failed. Please try:')
            print('   1. Wait a few hours and re-run this notebook')
            print('   2. Or manually download from https://deepmimo.net and upload to Drive')

## Step 4 — Verify Download

In [ ]:
import os

drive_zip = '/content/drive/MyDrive/DeepMIMO/o1_28_downloaded.zip'

if os.path.exists(drive_zip):
    size_mb = os.path.getsize(drive_zip) / (1024**2)
    print(f'✓ o1_28_downloaded.zip found on Drive')
    print(f'  Size: {size_mb:.1f} MB')
    if size_mb < 10:
        print('  ⚠️  File is very small — may be incomplete or an error page.')
    else:
        print('  ✓ File looks valid!')
        print()
        print('═' * 55)
        print('  NEXT STEPS — Copy to your local machine:')
        print('═' * 55)
        print()
        print('1. Go to Google Drive → DeepMIMO/')
        print('2. Download: o1_28_downloaded.zip')
        print('3. On your PC, place it at:')
        print('   c:\\Projects\\6G-ML\\deepmimo_scenarios\\')
        print()
        print('4. Extract it (it should create o1_28/ folder):')
        print('   Right-click → Extract Here')
        print()
        print('5. The final path should be:')
        print('   c:\\Projects\\6G-ML\\deepmimo_scenarios\\o1_28\\')
        print()
        print('6. Rename the folder if needed (remove _downloaded suffix)')
        print()
        print('7. Run training:')
        print('   python src/train.py --scenario o1_28 --device cpu')
else:
    print('❌ File not found on Drive. Check Step 3 output for errors.')

## Step 5 (Optional) — Quick Test in Colab
Verify the dataset loads correctly before downloading to your PC.

In [ ]:
import deepmimo as dm
import numpy as np

# Extract to Colab local storage for testing
import zipfile, os

LOCAL_EXTRACT = '/tmp/deepmimo_scenarios'
drive_zip = '/content/drive/MyDrive/DeepMIMO/o1_28_downloaded.zip'

if not os.path.exists(f'{LOCAL_EXTRACT}/o1_28'):
    os.makedirs(LOCAL_EXTRACT, exist_ok=True)
    print('Extracting zip for testing...')
    with zipfile.ZipFile(drive_zip, 'r') as z:
        z.extractall(LOCAL_EXTRACT)
    
    # Rename if needed (remove _downloaded suffix)
    for f in os.listdir(LOCAL_EXTRACT):
        if 'o1_28' in f and f != 'o1_28':
            os.rename(f'{LOCAL_EXTRACT}/{f}', f'{LOCAL_EXTRACT}/o1_28')
            print(f'  Renamed {f} → o1_28')
    print('✓ Extracted!')

# Load with DeepMIMO v4
dataset = dm.load('o1_28', scenario_dir=LOCAL_EXTRACT)

# Configure BS antenna
params = dm.ChannelParameters()
params.bs_antenna['shape'] = [64, 1]   # 64-element ULA
params.ue_antenna['shape'] = [1, 1]
params.ofdm['subcarriers'] = 64
params.ofdm['selected_subcarriers'] = [0]
params.num_paths = 10

channels = dataset.compute_channels(params)
positions = dataset.rx_pos

print(f'✓ Dataset loaded!')
print(f'  Total UE positions : {len(positions):,}')
print(f'  Channel shape      : {channels.shape}')
print(f'  Positions range X  : [{positions[:,0].min():.1f}, {positions[:,0].max():.1f}] m')
print(f'  Positions range Y  : [{positions[:,1].min():.1f}, {positions[:,1].max():.1f}] m')

# Show LOS statistics
norms = np.linalg.norm(channels.reshape(len(channels), -1), axis=1)
valid = (norms > 1e-6).sum()
print(f'  Valid (LOS) users  : {valid:,} / {len(positions):,} ({100*valid/len(positions):.1f}%)')